# Gemma: chuẩn bị, xác nhận và đánh giá trích xuất

Gemma `google/gemma-4-E4B-it` dùng để **trích xuất tin tuyển dụng**. Ridge/Random Forest dự đoán lương ở notebook 02. Notebook này chưa chạy train; Run All với mặc định chỉ kiểm tra và in kế hoạch.

Nhãn silver từ phiếu dữ liệu thật: 364/72/82 mẫu tương ứng 122/25/28 tin nguồn sau khử trùng prompt. Mẫu bỏ lương/kinh nghiệm là biến thể, không phải tin mới. Chưa đại diện cho tin tự do đầy đủ hoặc chất lượng tư vấn.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
assert (ROOT / "configs/gemma_train.yaml").exists()
PYTHON = sys.executable

def run_script(*args):
    subprocess.run([PYTHON, *args], cwd=ROOT, check=True)

print("Workspace:", ROOT)

## 1. Kiểm tra dữ liệu và trọng số
Chỉ xem ví dụ train. Không dùng test để điều chỉnh prompt, tham số hoặc chọn checkpoint. Trọng số lớn không lưu lên Git; xem manifest và hướng dẫn tải trong README.

In [ ]:
run_script("scripts/validate_sft.py", "--strict-extraction")
manifest = json.loads((ROOT / "data/finetune/manifest.json").read_text())
weights = json.loads((ROOT / "outputs/gemma_preparation/weights.json").read_text())
print(json.dumps(manifest["splits"], ensure_ascii=False, indent=2))
print(json.dumps(weights, ensure_ascii=False, indent=2))
example = json.loads((ROOT / "data/finetune/train.jsonl").read_text().splitlines()[0])
print(json.dumps(example, ensure_ascii=False, indent=2))

## 2. Xuất kế hoạch, chưa train
Trial đầu: LoRA trên MLX 4-bit, 4 lớp cuối, rank 8, batch 1, 100 iterations, lr 1e-4, context 1024, mask prompt và gradient checkpoint. 100 iterations không phải 100 epochs. Thông lượng/bộ nhớ khi train chưa được đo.

Console train sẽ ghi loss, validation loss, tokens/s và bộ nhớ theo khả năng MLX-LM. Lưu checkpoint mỗi 25 bước. Loss không thay thế metric trích xuất.

In [ ]:
run_script("scripts/gemma_train.py")

## 3. Chỉ khởi chạy sau khi bạn xác nhận
Chỉ điền chuỗi bên dưới sau khi đã đồng ý kế hoạch. Mặc định ô này bỏ qua huấn luyện. Log lưu tại `outputs/training_logs/gemma_<UTC>/`; adapter lưu thư mục riêng theo lượt để bảo toàn checkpoint.

In [ ]:
GEMMA_TRAINING_CONFIRMATION = ""  # Chỉ đổi sau khi chính bạn xác nhận.
if GEMMA_TRAINING_CONFIRMATION == "TOI_XAC_NHAN_TRAIN_GEMMA":
    run_script("scripts/gemma_train.py", "--confirm-training")
else:
    print("Chưa xác nhận: KHÔNG chạy Gemma training.")

## 4. So sánh base và adapter trên validation
Bật inference khi sẵn sàng; đây là suy luận, không train. Dùng thư mục output mới mỗi lượt. Chọn checkpoint theo validation, sau đó mới đánh giá cả base và adapter cố định trên test; không chọn lại bằng test.

Metric: JSON validity, schema validity, accuracy 8 trường và Skill micro F1. Trường thiếu trong dự đoán tính sai, không bỏ khỏi mẫu số. Numeric so theo đơn vị gốc, tolerance 1e-6; skills so theo tập, F1 không phân biệt hoa/thường. Nhãn silver và các biến thể phụ thuộc làm giới hạn diễn giải kết quả.

In [ ]:
RUN_INFERENCE = False
ADAPTER = ""  # Điền adapters/gemma-jobs/<run_id> sau khi train thành công.
EVAL_RUN = "trial_001"  # Đổi nếu thư mục đã tồn tại.
if RUN_INFERENCE:
    run_script("scripts/evaluate_gemma.py", "--generate", "--split", "valid",
               "--output", f"outputs/metrics/gemma_{EVAL_RUN}_base_valid")
    if ADAPTER:
        run_script("scripts/evaluate_gemma.py", "--generate", "--split", "valid",
                   "--adapter", ADAPTER,
                   "--output", f"outputs/metrics/gemma_{EVAL_RUN}_adapter_valid")
else:
    print("Chưa chạy suy luận/đánh giá Gemma.")

## 5. Đánh giá cuối và ghi báo cáo
Sau khi chốt adapter theo validation, chạy lại lệnh đánh giá với `--split test` cho cả base và adapter, giữ nguyên cấu hình. Dẫn `metrics.json`, `predictions.jsonl`, dataset checksum, model revision và log train vào báo cáo. Chưa có metric Gemma cho đến khi thực hiện bước này.

Không ghép nhãn lương năm/tháng hoặc ngoại tệ vào một đơn vị giả định. Không gọi chênh lệch hệ số Ridge là tác động nhân quả hay hứa tăng thu nhập.